# 問題
85で読み込んだ訓練データの一部（例えば冒頭の4事例）に対して、パディングなどの処理を行い、トークン列の長さを揃えてミニバッチを構成せよ。

In [1]:
# --- pathの準備 ---
path_dev = "./SST-2/dev.tsv"
path_train = "./SST-2/train.tsv"

def load_dataset(path):
  with open(path, 'r', encoding='utf-8') as f:
    final_list = []
    for row in f:
      row = row.strip().split("\t")
      tmp_dict = {}
      if not row:
          continue
      # カテゴリ宣言行
      if row[1] == "0" or row[1] == "1":
        sentence = row[0]
        tmp_dict["text"] = sentence
        tmp_dict["label"] = int(row[1])
        final_list.append(tmp_dict)
      else:
        print("0と1以外です：", row[1])
    return final_list

load_train = load_dataset(path_train)
load_dev = load_dataset(path_dev)

from transformers import AutoTokenizer

model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def load2tokenizer(load_list):
  token_text_list = []
  for i in range(len(load_list)):
    tmp_dict = {}
    text = load_list[i]["text"]
    token_text = tokenizer(text, return_tensors="pt")
    tmp_dict["token_text"] = token_text
    tmp_dict["label"] = load_list[i]["label"]
    token_text_list.append(tmp_dict)
  return token_text_list

token_train = load2tokenizer(load_train)
token_dev = load2tokenizer(load_dev)

0と1以外です： label
0と1以外です： label


/Users/nakamuratuzumi/anaconda3/envs/bert_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ===== pad 前に list 化して 1 次元(バッチ軸)を squeeze =====
import torch

# 1) 先頭4件だけ取り出す
subset = token_train[:4]  # 先頭4件
print("💫subsetの中身", subset)

# ★ここがポイント：各要素を「list（=可変長の生配列）」に戻す
enc_list = []
for ex in subset:
    enc = {}
    for k, v in ex["token_text"].items():
        # v は torch.Size([1, L]) になっているので、[L] にしてから list 化
        enc[k] = v.squeeze(0).tolist()
    enc_list.append(enc)
print("💫enc_listの中身", enc_list)

# pad してから一括でテンソル化
batch = tokenizer.pad(
    enc_list,
    padding="longest",
    return_tensors="pt"   # ここで初めてテンソル化
)
print("💫batchの中身", batch)

labels = torch.tensor([ex["label"] for ex in subset], dtype=torch.long)

print("input_ids shape:", batch["input_ids"].shape)        # 例: torch.Size([4, L_max])
print("attention_mask shape:", batch["attention_mask"].shape)
if "token_type_ids" in batch:
    print("token_type_ids shape:", batch["token_type_ids"].shape)
print("labels shape:", labels.shape)                       # torch.Size([4])

💫subsetの中身 [{'token_text': {'input_ids': tensor([[50281, 21179,   747,  4279,   621,   432,   253, 17087,  5085,   209,
         50282]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}, 'label': 0}, {'token_text': {'input_ids': tensor([[50281, 24634,   642, 19311,  1157,   760,  5188,  2149,   305,  3544,
           209, 50282]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}, 'label': 0}, {'token_text': {'input_ids': tensor([[50281,  3529, 14528,   697,  5810,   285,  3461,   684,  1633,  2581,
          5389,   670,  1966,  3753,   209, 50282]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}, 'label': 1}, {'token_text': {'input_ids': tensor([[50281,  2013,  1550, 23228, 10048,   281,  3464,   253,  1072,  4768,
           209, 50282]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}, 'label': 0}]
💫enc_listの中身 [{'input_ids': [50281, 21179, 747, 4279, 621, 432, 253, 17087, 5085, 209, 50282], 'attent

In [12]:
import torch

def mini_batch(subset):
    # 1️⃣ list化（[1, L] → [L]）
    enc_list = []
    for ex in subset:
        enc = {k: v.squeeze(0).tolist() for k, v in ex["token_text"].items()}
        enc_list.append(enc)

    # 2️⃣ 最長系列に合わせてパディングしてテンソル化（B×L_max）
    batch_enc = tokenizer.pad(enc_list, padding="longest", return_tensors="pt")

    # 3️⃣ ラベルもテンソル化
    labels = torch.tensor([ex["label"] for ex in subset], dtype=torch.long)

    # 4️⃣ サンプル単位に再構成 → 「1つのリスト」で返す
    final_list = []
    batch_size = labels.size(0)
    for i in range(batch_size):
        # 各キーについて i 番目サンプルを取り出す
        sample_enc = {k: v[i] for k, v in batch_enc.items()}
        tmp_dict = {"token_text": sample_enc, "label": labels[i].item()}
        final_list.append(tmp_dict)

    return final_list

In [13]:
mini_batch(token_train)

[{'token_text': {'input_ids': tensor([50281, 21179,   747,  4279,   621,   432,   253, 17087,  5085,   209,
           50282, 50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283,
           50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283,
           50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283,
           50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283,
           50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283,
           50283, 50283, 50283, 50283, 50283, 50283, 50283, 50283]),
   'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
           0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
           0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])},
  'label': 0},
 {'token_text': {'input_ids': tensor([50281, 24634,   642, 19311,  1157,   760,  5188,  2149,   305,  3544,
             209, 50282, 50283, 50283